# HMF validation: cloelib vs colossus

In this notebook, we validate the cloelib HMF implementations that have a counterpart in colossus.

The colossus multiplicity function is evaluated at the same $\sigma(M)$ computed by cloelib, so the comparison isolates the fitting function.

Tinker et al. (2008) halo mass function from cloelib compared with the reference implementation in [colossus](https://bdiemer.bitbucket.io/colossus/):\
Residual differences (zero for $M_{\rm 200m}$, below 0.5% otherwise, at the high-mass end) come from colossus rounding $\Delta_{\rm m}$ to an integer.

In [ ]:
%pip install --quiet colossus

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from colossus.cosmology import cosmology
from colossus.lss import mass_function

from cloelib.cosmology.camb_cosmology import CAMBBackground, CAMBLinearPerturbations
from cloelib.observables.halos.halo_abundance import TinkerHaloAbundance
from cloelib.observables.halos.halo_model_properties import HaloModelProperties

H0 = 67.7
h = H0 / 100.0
cosmo_pars = dict(
    H0=H0,
    Omega_cdm0=0.12 / h**2,
    Omega_b0=0.022 / h**2,
    Omega_k0=0.0,
    w0=-1.0,
    wa=0.0,
    ns=0.96,
    mnu=0.06,
    As=2e-9,
    gamma_MG=0.0,
    N_mnu=1,
)
background = CAMBBackground(**cosmo_pars)
perturbations = CAMBLinearPerturbations(background, np.linspace(0.0, 2.0, 100))
halo_model_properties = HaloModelProperties(perturbations)

# colossus: same (cdm + baryons) cosmology
cosmology.setCosmology(
    "cloelib",
    dict(
        flat=True,
        H0=H0,
        Om0=halo_model_properties.Omega_cb_0,
        Ob0=cosmo_pars["Omega_b0"],
        sigma8=0.81,
        ns=cosmo_pars["ns"],
        relspecies=False,
    ),
)

In [ ]:
z = np.array([0.0, 0.5, 1.0])
M = np.logspace(10, 15, 101)  # M_sun / h
mass_definitions = {
    "vir": ("vir", 200),
    "200m": ("mean", 200),
    "200c": ("crit", 200),
}

plt.rcParams.update({"font.size": 15})

fig, axes = plt.subplots(2, len(mass_definitions), figsize=(16, 8), sharex=True,
                         sharey="row", gridspec_kw={"height_ratios": [2, 1]})

for col, (mdef, (overdensity_type, overdensity)) in enumerate(mass_definitions.items()):

    tinker = TinkerHaloAbundance(halo_model_properties, overdensity_type, overdensity)
    sigma = tinker.sigma_z_M(z, M)
    f_colossus = np.array([mass_function.modelTinker08(sigma[i], zz, mdef) for i, zz in enumerate(z)])

    dndlnM_cloelib = M * tinker.dn_dm(z, M)
    dndlnM_colossus = M * tinker.dn_dm_fsigmanu(z, M, f_colossus)

    for i, zz in enumerate(z):
        line, = axes[0, col].loglog(M, dndlnM_cloelib[i], label=f"cloelib, z={zz:g}")
        axes[0, col].loglog(M, dndlnM_colossus[i], ls="--", color="k", lw=0.8,
                            label="colossus" if i == 0 else None)
        axes[1, col].semilogx(M, 100 * (dndlnM_cloelib[i] / dndlnM_colossus[i] - 1), color=line.get_color())

    axes[0, col].set_title(rf"Tinker08, $M_{{\rm {mdef}}}$")
    axes[1, col].axhline(0, color="k", lw=0.8)
    axes[1, col].set_xlabel(r"$M\ [\mathrm{M}_\odot/h]$")

axes[0, 0].set_ylabel(r"$dn/d\ln M\ [h^3\,\mathrm{Mpc}^{-3}]$")
axes[1, 0].set_ylabel("cloelib / colossus - 1 [%]")
axes[1, 0].set_ylim(-2, 2)
handles, labels = axes[0, 0].get_legend_handles_labels()
order = [labels.index("colossus")] + [j for j, label in enumerate(labels) if label != "colossus"]
axes[0, 0].legend([handles[j] for j in order], [labels[j] for j in order])
fig.tight_layout()
plt.show()